In [141]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [142]:
training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [143]:
batch_size = 32

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([32, 3, 32, 32])
Shape of y: torch.Size([32]) torch.int64


In [144]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

Using cuda device


In [ ]:
# Define model
class ConvNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 32×32 → 16×16
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),          
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.4),     
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model10 = ConvNet(num_classes=10).to(device)
print(model10)

ConvNet(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.2, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (15): Dropout(p=0.3, inplace=F

In [146]:
loss_fn = nn.CrossEntropyLoss()
optimizer10 = torch.optim.RAdam(model10.parameters(), lr=0.001)
# Adadelta 
# Adafactor 
# Adagrad 
# Adam 
# AdamW Implements AdamW algorithm, where weight decay does not accumulate in the momentum nor variance.
# SparseAdam SparseAdam implements a masked version of the Adam algorithm suitable for sparse gradients.
# Adamax Implements Adamax algorithm (a variant of Adam based on infinity norm).
# ASGD Implements Averaged Stochastic Gradient Descent.     skoro SGD działa kiepsko to nie próbowałbym tego
# LBFGS 
# Muon 
# NAdam - nieźle, ale ADAM lepiej
# RAdam 
# RMSprop 
# Rprop 
# SGD stochastic gradient descent                           działa kiepsko na tych

# lista schedulerów:
# StepLR — redukuje lr o stałą wartość co x epok
# MultiStepLR — jak StepLR, ale redukcja w określonych z góry epokach (np. 10., 15., 20.)
# ExponentialLR — nie zagłębiam się w działaniu, płynniejszy niż poprzednie
# ConstantLR — raz zmniejsza/zwiększa, np. 0.01 0.01 0.01 0.005 0.005 0.005
# LinearLR — nieciekawe
# PolynomialLR - zmnienia wielomianowo
# CosineAnnealingLR — obniża lr wzdłuż krzywej cosinus do minimum, potem restart
# CosineAnnealingWarmRestarts — jak wyżej, ale z automatycznymi restartami co n epok gdyby nie osiągneło minimum
# CyclicLR
# OneCycleLR
# ReduceLROnPlateau — redukuje lr gdy loss (przy mode=min) nie spada lub accuracy nie rośnie
from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(
    optimizer10,
    mode='max',        
    factor=0.3,        
    patience=2,        
    min_lr=0.00001,     
    threshold=0.01,   
    cooldown=1         
)

In [147]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss_value, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss_value:>7f}  [{current:>5d}/{size:>5d}]")

In [148]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    accuracy = 100 * correct
    print(f"Test Error: \n Accuracy: {accuracy:>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return accuracy, test_loss

In [149]:
epochs = 25
training_log = []

for t in range(epochs):
    print(f"Epoch {t+1} (CIFAR-10)\n-------------------------------")
    train(train_dataloader, model10, loss_fn, optimizer10)
    accuracy, avg_loss = test(test_dataloader, model10, loss_fn)
    training_log.append({
        "epoch": t + 1,
        "accuracy": accuracy,
        "avg_loss": avg_loss,
    })
    lr_before = optimizer10.param_groups[0]['lr']
    scheduler.step(avg_loss)
    lr_after = optimizer10.param_groups[0]['lr']

    if lr_after != lr_before:
        print(f"Epoch {t+1}: lr zmienione {lr_before:.6f} → {lr_after:.6f}")
    else:
        print(f"Epoch {t+1}: lr = {lr_after:.6f}")
print("Done CIFAR-10")

Epoch 1 (CIFAR-10)
-------------------------------
loss: 2.296188  [   32/50000]
loss: 1.963313  [ 3232/50000]
loss: 1.870357  [ 6432/50000]
loss: 1.765044  [ 9632/50000]
loss: 1.506415  [12832/50000]
loss: 1.398587  [16032/50000]
loss: 1.253462  [19232/50000]
loss: 1.101972  [22432/50000]
loss: 1.407523  [25632/50000]
loss: 2.256556  [28832/50000]
loss: 1.458086  [32032/50000]
loss: 1.653949  [35232/50000]
loss: 1.184273  [38432/50000]
loss: 1.458108  [41632/50000]
loss: 1.402886  [44832/50000]
loss: 1.098351  [48032/50000]
Test Error: 
 Accuracy: 57.6%, Avg loss: 1.220394 

Epoch 1: lr = 0.001000
Epoch 2 (CIFAR-10)
-------------------------------
loss: 1.490089  [   32/50000]
loss: 1.384085  [ 3232/50000]
loss: 1.091359  [ 6432/50000]
loss: 1.232391  [ 9632/50000]
loss: 1.067776  [12832/50000]
loss: 0.952071  [16032/50000]
loss: 0.943699  [19232/50000]
loss: 0.994823  [22432/50000]
loss: 0.935704  [25632/50000]
loss: 1.263248  [28832/50000]
loss: 1.151576  [32032/50000]
loss: 0.63736

KeyboardInterrupt: 

In [ ]:
import json
import os

models_dir = "models10"
meta_dir="metadata10"

# sprawdzamy ile jest plików w folderze
n = len([name for name in os.listdir(models_dir) if os.path.isfile(os.path.join(models_dir, name))])

# jeśli jest n modeli, to nowy będzie cifar10_(n+1)
model_path = os.path.join(models_dir, f"cifar10_{n+1}.pth")
torch.save(model10.state_dict(), model_path)

training_metadata = {
    "network_architecture": str(model10),
    "optimizer": optimizer10.__class__.__name__,
    "learning_rate": optimizer10.param_groups[0].get("lr"),
    "scheduler": scheduler.__class__.__name__,
    "scheduler_params": {
        "mode": scheduler.mode,
        "factor": scheduler.factor,
        "patience": scheduler.patience,
        "min_lr": scheduler.min_lrs[0] if hasattr(scheduler, "min_lrs") else None,
        "threshold": scheduler.threshold,
        "cooldown": scheduler.cooldown,
    },
    "batch_size": batch_size,
    "training_log": training_log,
}

metadata_path = os.path.join(meta_dir, f"{n+1}.txt")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, ensure_ascii=False, indent=2)

print(f"Found {n} files in {models_dir}.")
print(f"Saved model to {model_path}")
print(f"Saved training metadata to {metadata_path}")

Found 5 files in models10.
Saved model to models10\cifar10_6.pth
Saved training metadata to metadata10\6.txt
